# Pratica 01 — Che cos'è l'NLP

**Corso:** NLP Classico con Python · **Lezione 01** · **Tempo stimato:** 45-60 minuti
**Notebook di lezione:** `notebooks_clase/clase01_que_es_el_nlp_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Tutti i testi sono nella cella di dati che segue il setup. Sono brevi e scritti per il corso (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Non serve nessuna libreria oltre a Python.

---
## Cella 0 — Setup e dati

In [ ]:
import re
from collections import Counter

print("Librerie caricate correttamente.")

In [ ]:
# datos-es
RAIZ_ES = "habl"
TER_ES = {"pres": ["o", "as", "a", "amos", "áis", "an"], "imp": ["aba", "abas", "aba", "ábamos", "abais", "aban"], "fut": ["aré", "arás", "ará", "aremos", "aréis", "arán"]}
RAIZ_IT = "parl"
TER_IT = {"pres": ["o", "i", "a", "iamo", "ate", "ano"], "imp": ["avo", "avi", "ava", "avamo", "avate", "avano"], "fut": ["erò", "erai", "erà", "eremo", "erete", "eranno"]}

FRASES_AMBIGUAS = {
    1: "Vi al hombre con el telescopio",
    2: "Me senté en el banco del parque",
    3: "Ho comprato una pesca al mercato",
    4: "Ho visto la ragazza con il binocolo",
}

POSITIVAS = {"spa": {"bueno", "buena", "excelente"}, "ita": {"buono", "buona", "ottimo"}}
NEGATIVAS = {"spa": {"malo", "mala", "horrible"}, "ita": {"cattivo", "pessimo", "orribile"}}
NEGADORES = {"spa": {"no", "nunca"}, "ita": {"non", "mai"}}
CASOS_NEGACION = [
    ("spa", "el producto es bueno", "pos"),
    ("spa", "el producto no es bueno", "neg"),
    ("spa", "nunca fue malo", "pos"),
    ("spa", "servicio horrible", "neg"),
    ("spa", "todo normal", None),
    ("ita", "il servizio è buono", "pos"),
    ("ita", "il servizio non è buono", "neg"),
    ("ita", "non è orribile", "pos"),
    ("ita", "prodotto orribile", "neg"),
    ("ita", "tutto normale", None),
]

PERFILES = {
    "spa": {"el", "los", "las", "de", "que", "y", "en", "es", "por", "con", "para", "pero", "mi", "muy", "no", "hay", "está", "estoy", "son", "se", "nada", "cuando", "esta", "tienda", "tiene", "necesito", "otra", "quiero"},
    "ita": {"il", "lo", "i", "gli", "le", "di", "che", "e", "in", "è", "per", "con", "ma", "mio", "molto", "non", "sono", "si", "nel", "niente", "quando", "questo", "c", "ho", "ne", "serve", "altra", "vorrei", "hanno"},
}
FRASES_IDIOMA = [
    ("El servicio es muy bueno y la entrega fue rápida", "spa"),
    ("No hay nada en mi cuenta y necesito ayuda", "spa"),
    ("Los productos de esta tienda son caros pero de buena calidad", "spa"),
    ("La aplicación se cierra cuando intento pagar con la tarjeta", "spa"),
    ("Estoy muy contento con el envío y con la atención", "spa"),
    ("Il servizio è molto buono e la consegna è stata rapida", "ita"),
    ("Non c'è niente nel mio conto e ho bisogno di aiuto", "ita"),
    ("I prodotti di questo negozio sono cari ma di buona qualità", "ita"),
    ("L'applicazione si chiude quando provo a pagare con la carta", "ita"),
    ("Sono molto contento della spedizione e dell'assistenza", "ita"),
]

CATEGORIAS = {
    "spa": {"fact": {"cobraron", "tarjeta", "reembolso", "factura", "importe", "cobro", "pago"},
            "env": {"paquete", "pedido", "llegó", "llegado", "envíen", "retraso", "entrega"},
            "tec": {"aplicación", "app", "error", "contraseña", "conectar", "funciona", "cuenta"}},
    "ita": {"fact": {"addebitato", "carta", "rimborso", "fattura", "importo"},
            "env": {"pacco", "consegna", "ritardo", "arrivato", "ordine", "spedizione"},
            "tec": {"app", "errore", "password", "funziona", "account", "collegarmi"}},
}
TICKETS_PRUEBA = [
    ("Me cobraron dos veces con la tarjeta y quiero un reembolso", "spa", "fact"),
    ("El paquete no ha llegado y el pedido lleva diez días de retraso", "spa", "env"),
    ("La aplicación muestra un error y no puedo conectar con mi cuenta", "spa", "tec"),
    ("La factura tiene un importe equivocado, necesito otra", "spa", "fact"),
    ("Mi pedido llegó roto y quiero que me envíen otro paquete", "spa", "env"),
    ("No funciona la contraseña nueva, la app da error", "spa", "tec"),
    ("Mi hanno addebitato due volte sulla carta e voglio il rimborso", "ita", "fact"),
    ("Il pacco non è arrivato e la consegna è in ritardo di dieci giorni", "ita", "env"),
    ("L'app mostra un errore e non riesco a collegarmi al mio account", "ita", "tec"),
    ("La fattura ha un importo sbagliato, ne serve un'altra", "ita", "fact"),
    ("Il mio ordine è arrivato rotto e vorrei un altro pacco", "ita", "env"),
    ("La nuova password non funziona, l'app dà errore", "ita", "tec"),
]

## Esercizio 1 — Contare le forme di un verbo

Con le terminazioni di `TER_ES` (verbo `hablar`) e `TER_IT` (verbo `parlare`) per tre tempi (presente, imperfetto e futuro, sei persone ciascuno), calcola:

- `formas_es`, `formas_it`: la lista delle 18 forme di ciascuna lingua (radice + terminazione),
- `distintas_es`, `distintas_it`: quante forme **distinte** ci sono,
- `repetidas_es`: l'**insieme** delle forme che compaiono in più di un posto (in italiano non ce n'è nessuna).

Prima di programmare, verifica a mano se in spagnolo coincide qualche forma tra le 18.

In [ ]:
formas_es = None      # TODO: lista con radice + terminazione per i tre tempi
formas_it = None      # TODO
distintas_es = None   # TODO
distintas_it = None   # TODO
repetidas_es = None   # TODO: insieme delle forme con più di un'occorrenza

# Controllo automatico
assert all(v is not None for v in (formas_es, formas_it, distintas_es, distintas_it, repetidas_es)), "Completa le righe contrassegnate con TODO"
assert len(formas_es) == 18 and len(formas_it) == 18, "ogni lingua ha 3 tempi × 6 persone = 18 forme"
assert distintas_es == 17 and distintas_it == 18, "in spagnolo una forma si ripete nell'imperfetto; in italiano non se ne ripete nessuna"
assert repetidas_es == {RAIZ_ES + "aba"}, "la forma ripetuta è la prima e la terza persona dell'imperfetto"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Tipi di ambiguità

In `FRASES_AMBIGUAS` hai quattro frasi. Classifica ciascuna in base al suo tipo di ambiguità con il codice: `"lex"` (lessicale: una parola con più sensi) o `"syn"` (sintattica: due modi di raggruppare le parole). Compila il dizionario `tipo_ambiguedad` (chiave = numero della frase).

In [ ]:
tipo_ambiguedad = {1: None, 2: None, 3: None, 4: None}      # TODO: "lex" o "syn" per ogni frase

# Controllo automatico
assert all(v is not None for v in tipo_ambiguedad.values()), "Completa le righe contrassegnate con TODO"
assert set(tipo_ambiguedad.values()) <= {"lex", "syn"}, "usa solo i codici lex e syn"
assert tipo_ambiguedad[1] == "syn" and tipo_ambiguedad[4] == "syn", "«con il telescopio» / «con il binocolo»: a quale parola si collega il gruppo?"
assert tipo_ambiguedad[2] == "lex" and tipo_ambiguedad[3] == "lex", "banco e pesca hanno più sensi"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Un classificatore a regole con negazione

Completa la funzione `clasificar` in modo che un negatore inverta la polarità della parola con carica che compare fino a `ventana` posizioni dopo. La funzione restituisce `"pos"`, `"neg"` o `None` in caso di parità (o se non c'è nessuna parola con carica). Si prova con dieci frasi nelle due lingue (`CASOS_NEGACION`).

In [ ]:
def clasificar(texto, idioma, ventana=2):
    palabras = re.findall(r"\w+", texto.lower())
    puntos = 0
    for i, palabra in enumerate(palabras):
        valor = 1 if palabra in POSITIVAS[idioma] else -1 if palabra in NEGATIVAS[idioma] else 0
        hay_negador = None      # TODO: c'è un negatore nelle `ventana` parole precedenti?
        if hay_negador:
            valor = -valor
        puntos += valor
    return "pos" if puntos > 0 else "neg" if puntos < 0 else None

# Controllo automatico
for idioma, texto, esperado in CASOS_NEGACION:
    assert clasificar(texto, idioma) == esperado, f"rivedi la negazione in: {texto!r} (atteso {esperado})"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Ordinare una pipeline

Un negozio vuole classificare i suoi ticket di assistenza per categoria con un modello di apprendimento. Ecco i passi, in disordine:

`PASOS = ["rappresentare", "valutare", "pulire", "addestrare", "acquisire", "tokenizzare", "dividere"]`

Scrivi in `orden` una lista con i sette passi in un ordine **valido**. Esiste più di un ordine possibile; il controllo richiede solo le dipendenze: prima si acquisiscono i dati, si puliscono, si tokenizzano; l'insieme di test si separa **prima di rappresentare** (per non usare informazione del test quando si costruisce il vocabolario); si rappresenta prima di addestrare e si addestra prima di valutare.

In [ ]:
PASOS = ["rappresentare", "valutare", "pulire", "addestrare", "acquisire", "tokenizzare", "dividere"]
orden = None      # TODO: lista con i sette passi in un ordine valido

# Controllo automatico
assert orden is not None, "Completa le righe contrassegnate con TODO"
assert sorted(orden) == sorted(PASOS), "usa ogni passo esattamente una volta"
RESTRICCIONES = [("acquisire", "pulire"), ("pulire", "tokenizzare"), ("tokenizzare", "rappresentare"),
                 ("dividere", "rappresentare"), ("rappresentare", "addestrare"), ("addestrare", "valutare")]
for antes, despues in RESTRICCIONES:
    assert orden.index(antes) < orden.index(despues), f"'{antes}' deve venire prima di '{despues}'"
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Un rilevatore di italiano e spagnolo

`PERFILES` contiene, per ogni lingua (`"spa"`, `"ita"`), un insieme di parole molto frequenti (un profilo di stopword). Completa `detectar_idioma`: tokenizza il testo in minuscolo con `re.findall(r"\w+", ...)`, conta quante parole sono in ciascun profilo e restituisci la lingua con più corrispondenze, oppure `None` in caso di parità (per esempio un testo vuoto). Si prova con dieci frasi (`FRASES_IDIOMA`).

> **Nota:** i profili e le parole chiave sono stati scelti a mano per questo mini esercizio e si provano sulle stesse frasi che vedi. Serve per esercitare la logica, **non come valutazione reale**: un rilevatore serio si valuta con dati separati da quelli usati per costruirlo (lo faremo a partire dalla lezione 5).

In [ ]:
def detectar_idioma(texto):
    palabras = re.findall(r"\w+", texto.lower())
    aciertos_spa = None      # TODO: quante parole sono nel profilo "spa"
    aciertos_ita = None      # TODO: quante parole sono nel profilo "ita"
    if aciertos_spa is None or aciertos_ita is None:
        return None
    return "spa" if aciertos_spa > aciertos_ita else "ita" if aciertos_ita > aciertos_spa else None

# Controllo automatico
assert detectar_idioma("") is None, "un testo vuoto non ha lingua"
aciertos = sum(detectar_idioma(texto) == idioma for texto, idioma in FRASES_IDIOMA)
assert aciertos == len(FRASES_IDIOMA), f"ne indovina {aciertos} su {len(FRASES_IDIOMA)}: rivedi il conteggio"
print(f"Esercizio 5 corretto ✔ ({aciertos} su {len(FRASES_IDIOMA)} frasi)")

---
## Sfida finale — Lingua e categoria di un ticket

Integra quanto visto: scrivi `analizar_ticket(texto)`, che restituisce una coppia `(idioma, categoria)`.

- La lingua si decide con i profili di `PERFILES` (come nell'esercizio 5).
- La categoria (`"fact"` = fatturazione, `"env"` = spedizione, `"tec"` = problema tecnico) si decide con le parole chiave di `CATEGORIAS[idioma]`: vince la categoria con più parole chiave presenti; `None` in caso di parità.

Si valuta su dodici ticket di test (`TICKETS_PRUEBA`, sei per lingua; *esempio illustrativo*) e si richiede un'accuratezza minima di 11 su 12 sia nella lingua sia nella categoria.

> **Nota:** i profili e le parole chiave sono stati scelti a mano per questo mini esercizio e si provano sulle stesse frasi che vedi. Serve per esercitare la logica, **non come valutazione reale**: un rilevatore serio si valuta con dati separati da quelli usati per costruirlo (lo faremo a partire dalla lezione 5).

In [ ]:
def analizar_ticket(texto):
    palabras = re.findall(r"\w+", texto.lower())
    # TODO 1: decidi la lingua contando le parole in ogni profilo
    idioma = None
    # TODO 2: con le parole chiave di CATEGORIAS[idioma], scegli la categoria con più corrispondenze
    categoria = None
    return idioma, categoria

# Controllo automatico
resultados_idioma = []
resultados_categoria = []
for texto, idioma_real, categoria_real in TICKETS_PRUEBA:
    idioma, categoria = analizar_ticket(texto)
    resultados_idioma.append(idioma == idioma_real)
    resultados_categoria.append(categoria == categoria_real)
assert sum(resultados_idioma) >= 11, f"lingua: {sum(resultados_idioma)} su 12"
assert sum(resultados_categoria) >= 11, f"categoria: {sum(resultados_categoria)} su 12"
print(f"Sfida finale corretta ✔ (lingua {sum(resultados_idioma)}/12, categoria {sum(resultados_categoria)}/12)")

**Le tue conclusioni (scrivi 3-4 frasi):** cosa fallisce nel tuo rilevatore di lingua con testi molto brevi o con parole condivise tra italiano e spagnolo? Cosa succederebbe alle regole di categoria con un ticket che mescola due problemi, o con una parola scritta male? In quale punto della pipeline correggeresti ciascun guasto?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
formas_es = [RAIZ_ES + t for lista in TER_ES.values() for t in lista]
formas_it = [RAIZ_IT + t for lista in TER_IT.values() for t in lista]
distintas_es = len(set(formas_es))
distintas_it = len(set(formas_it))
repetidas_es = {f for f, n in Counter(formas_es).items() if n > 1}

### Soluzione dell'esercizio 2

In [ ]:
tipo_ambiguedad = {1: "syn", 2: "lex", 3: "lex", 4: "syn"}

assert all(v is not None for v in tipo_ambiguedad.values()), "Completa le righe contrassegnate con TODO"

### Soluzione dell'esercizio 3

In [ ]:
def clasificar(texto, idioma, ventana=2):
    palabras = re.findall(r"\w+", texto.lower())
    puntos = 0
    for i, palabra in enumerate(palabras):
        valor = 1 if palabra in POSITIVAS[idioma] else -1 if palabra in NEGATIVAS[idioma] else 0
        hay_negador = any(p in NEGADORES[idioma] for p in palabras[max(0, i - ventana):i])
        if hay_negador:
            valor = -valor
        puntos += valor
    return "pos" if puntos > 0 else "neg" if puntos < 0 else None

for idioma, texto, esperado in CASOS_NEGACION:
    assert clasificar(texto, idioma) == esperado, f"rivedi la negazione in: {texto!r} (atteso {esperado})"

### Soluzione dell'esercizio 4

In [ ]:
PASOS = ["rappresentare", "valutare", "pulire", "addestrare", "acquisire", "tokenizzare", "dividere"]
orden = ["acquisire", "pulire", "tokenizzare", "dividere", "rappresentare", "addestrare", "valutare"]

assert orden is not None, "Completa le righe contrassegnate con TODO"

### Soluzione dell'esercizio 5

In [ ]:
def detectar_idioma(texto):
    palabras = re.findall(r"\w+", texto.lower())
    aciertos_spa = sum(p in PERFILES["spa"] for p in palabras)
    aciertos_ita = sum(p in PERFILES["ita"] for p in palabras)
    if aciertos_spa is None or aciertos_ita is None:
        return None
    return "spa" if aciertos_spa > aciertos_ita else "ita" if aciertos_ita > aciertos_spa else None

aciertos = sum(detectar_idioma(texto) == idioma for texto, idioma in FRASES_IDIOMA)

### Soluzione della sfida finale

In [ ]:
def analizar_ticket(texto):
    palabras = re.findall(r"\w+", texto.lower())
    puntos_idioma = {i: sum(p in perfil for p in palabras) for i, perfil in PERFILES.items()}
    mejor = max(puntos_idioma.values())
    ganadores = [i for i, n in puntos_idioma.items() if n == mejor]
    idioma = ganadores[0] if len(ganadores) == 1 else None
    if idioma is None:
        return None, None
    puntos_cat = {c: sum(p in claves for p in palabras) for c, claves in CATEGORIAS[idioma].items()}
    mejor_cat = max(puntos_cat.values())
    ganadoras = [c for c, n in puntos_cat.items() if n == mejor_cat]
    categoria = ganadoras[0] if mejor_cat > 0 and len(ganadoras) == 1 else None
    return idioma, categoria

resultados_idioma = []
resultados_categoria = []
for texto, idioma_real, categoria_real in TICKETS_PRUEBA:
    idioma, categoria = analizar_ticket(texto)
    resultados_idioma.append(idioma == idioma_real)
    resultados_categoria.append(categoria == categoria_real)
assert sum(resultados_idioma) >= 11, f"lingua: {sum(resultados_idioma)} su 12"
assert sum(resultados_categoria) >= 11, f"categoria: {sum(resultados_categoria)} su 12"
print(f"Sfida finale corretta ✔ (lingua {sum(resultados_idioma)}/12, categoria {sum(resultados_categoria)}/12)")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (pipeline, tipi di ambiguità, morfologia, clitici, classico contro neurale)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*